## Background 
This notebook showcases how to leverage Optuna for hyperparameter tuning, specifically for the n_lists and n_probes parameters. We will demonstrate how to optimize these parameters using Optuna's Bayesian optimization capabilities.

In [ ]:
#Install Required Packages
!python -m pip install optuna

In [ ]:
import cupy as cp
import numpy as np
from cuvs.neighbors import ivf_flat
import urllib.request
import time
import optuna
from utils import calc_recall
import os

## Download wiki-all dataset


In [ ]:
import tarfile
home_dir = os.path.expanduser("~/")
#wiki-all datasets are in tar format
def download_files(url, file):
    if os.path.exists(home_dir + "/" + file):
        print("tar file is already downloaded")
    else:
        urllib.request.urlretrieve(url, home_dir + "/" + file)
    # Open the .tar file
    with tarfile.open(home_dir + "/" + file, 'r') as tar:
        filename = file.split(".")[0]
        if os.path.exists(home_dir + "/" + filename + "/"):
            print("Files already extracted")
            return home_dir + "/" + filename + "/"
        # Extract all contents into the specified directory
        extract_path=home_dir + "/" +file.split(".")[0]
        tar.extractall(extract_path)
    return extract_path

In [ ]:
extracted_path=download_files('https://data.rapids.ai/raft/datasets/wiki_all_1M/wiki_all_1M.tar', 'wiki_all_1M.tar')

## Dataset Preparation: Load fbin, ibin files 
This example utilizes the Wiki-1M dataset, a collection of four binary files containing: 

Database vectors: Used for index building and searching.
Query vectors: Used for index building and searching.
Ground truth neighbors: Associated with a particular distance, used for evaluation.
Distances: Associated with a particular distance, used for evaluation.
The file suffixes denote the data type of vectors stored in the file: 

.fbin: float32
.ibin: int
For more information on the Wiki-1M dataset, please refer to the [RAPIDS documentation](https://docs.rapids.ai/api/raft/nightly/ann_benchmarks_dataset)
.

In [ ]:
def read_data(file_path, dtype):
    with open(file_path, "rb") as f:
        rows,cols = np.fromfile(f, count=2, dtype= np.int32)
        d = np.fromfile(f,count=rows*cols,dtype=dtype).reshape(rows, cols)
    return cp.asarray(d)

In [ ]:
vectors= read_data(extracted_path + "/base.1M.fbin",np.float32)
queries = read_data(extracted_path + "/queries.fbin",np.float32)
gt_neighbors = read_data(extracted_path + "/groundtruth.1M.neighbors.ibin",np.int32)

In [ ]:
#Get the dataset size of database vectors
dataset_size = vectors.shape[0]
dim = vectors.shape[1]

## Visualization

Generates and displays Pareto front plots for a given Optuna study object.

In [ ]:
def visualization(study_obj):
    """
    This function creates two Pareto front plots to visualize trade-offs between different
    optimization objectives. The plots help in understanding the balance between competing
    objectives in the optimization process.

    Args:
        study_obj (optuna.Study): The Optuna study object containing the optimization results.

    The function produces the following plots:
    1. **Figure 1**: A Pareto front plot showing the trade-off between `build_time_in_secs`
       and `recall`. It visualizes how the optimization process balances the build time
       and recall score.
    2. **Figure 2**: A Pareto front plot showing the trade-off between `latency_in_ms`
       and `recall`. This plot illustrates the relationship between latency and recall score.

    """

    fig1 = optuna.visualization.plot_pareto_front(
    study_obj,
    targets=lambda t: (t.values[0], t.values[2]),
    target_names=["build_time_in_secs", "recall"],
    )
    fig1.show()

    fig2 = optuna.visualization.plot_pareto_front(
        study_obj,
        targets=lambda t: (t.values[1], t.values[2]),
        target_names=["latency_in_ms", "recall"],
    )
    fig2.show()

In [ ]:
def print_target_instance_summary(target_instance):
    print(f"\tnumber: {target_instance.number}")
    print(f"\tparams: {target_instance.params}")
    print(f"\tvalues: {target_instance.values}")

def print_best_trial_values(optuna_study):
    """
    Prints details about the trials on the Pareto front of an Optuna study.

    This function analyzes the best trials from an Optuna study, which are typically
    those with the most favorable trade-offs among multiple objectives. It prints
    information on three specific metrics:

    1. The number of trials on the Pareto front.
    2. The trial with the highest accuracy among the best trials.
    3. The trial with the lowest build time among the best trials.
    4. The trial with the lowest latency among the best trials.

    Parameters:
    optuna_study (optuna.study.Study): An Optuna study object that contains information
    about the trials and their respective metrics.

    The function assumes that each trial has three metrics recorded in the `values` list:
    - `values[0]`: Build time
    - `values[1]`: latency
    - `values[2]`: Accuracy

    """
    print(f"Number of trials on the Pareto front: {len(optuna_study.best_trials)}")

    trial_with_lowest_build_time = min(optuna_study.best_trials, key=lambda t: t.values[0])
    print(f"Trial with lowest build time in secs: ")
    print_target_instance_summary(trial_with_lowest_build_time)

    trial_with_lowest_latency = min(optuna_study.best_trials, key=lambda t: t.values[1])
    print(f"Trial with lowest latency in ms: ")
    print_target_instance_summary(trial_with_lowest_latency)

    trial_with_highest_accuracy = max(optuna_study.best_trials, key=lambda t: t.values[2])
    print(f"Trial with highest accuracy: ")
    print_target_instance_summary(trial_with_highest_accuracy)

## Hyperparameter Optimization (HPO) for CUVS Libraries

An Optuna trial object used to suggest values for the hyperparameters of various CUVS libraries (such as ivf_flat, ivf_pq, and cagra).

The multi-objective function returns a tuple of three float values, each rounded to four decimal places:

build_time_in_secs: Time taken to build the index, measured in seconds.
latency_in_ms: Average search latency, measured in milliseconds. Calculated as the total search time divided by the number of queries.
recall: Recall metric, indicating the proportion of relevant neighbors retrieved.


## ivf_flat HPO example

In [ ]:
def multi_objective_ivf_flat(trial):
    """
    Optimizes the parameters for an Inverted File Index (IVF) Flat index in a multi-objective setting.

    """
    # Suggest an integer for the number of lists
    n_lists = trial.suggest_int("n_lists", 10, dataset_size*0.1)
    # Suggest an integer for the number of probes
    n_probes = trial.suggest_int("n_probes",n_lists*0.01 , n_lists*0.1)
    build_params = ivf_flat.IndexParams(
        n_lists=n_lists,
    )
    start_build_time = time.time()
    index = ivf_flat.build(build_params, vectors)
    build_time_in_secs = time.time() - start_build_time

    # Configure search parameters
    search_params = ivf_flat.SearchParams(n_probes=n_probes)
    # Perform the search
    start_search_time = time.time()
    distances, indices = ivf_flat.search(search_params, index, queries, k=10)
    search_time = time.time() - start_search_time

    latency_in_ms = (search_time * 1000)/queries.shape[0]

    found_distances, found_indices = cp.asnumpy(distances), cp.asnumpy(indices)
    recall = calc_recall(found_indices, gt_neighbors)
    return round(build_time_in_secs,4), round(latency_in_ms,4), round(recall,4)

In [ ]:
ivf_flat_study = optuna.create_study(directions=['minimize', 'minimize', 'maximize'])
ivf_flat_study.optimize(multi_objective_ivf_flat, n_trials=10)

In [ ]:
print_best_trial_values(ivf_flat_study)

In [ ]:
visualization(ivf_flat_study)

## ivf_pq HPO example

In [ ]:
from cuvs.neighbors import ivf_pq,refine

In [ ]:
def multi_objective_ivf_pq(trial):
    """
    Optimizes hyperparameters for Inverted File Product Quantization (IVF-PQ) in a multi-objective setting..

    """
    # Suggest values for build parameters
    pq_dim = trial.suggest_int("pq_dim", dim*0.25, dim, step=2)
    n_lists = 1000

    # Suggest an integer for the number of probes
    n_probes = trial.suggest_int("n_probes",n_lists*0.01 , n_lists*0.1)

    build_params = ivf_pq.IndexParams(
    n_lists=n_lists,
    pq_dim=pq_dim,
    )

    start_build_time = time.time()
    index = ivf_pq.build(build_params, vectors)
    build_time_in_secs = time.time() - start_build_time

    # Configure search parameters
    search_params = ivf_pq.SearchParams(n_probes=n_probes)

    # perform search and refine to increase recall/accuracy
    start_search_time = time.time()
    distances, indices = ivf_pq.search(search_params, index, queries, k=10)
    search_time = time.time() - start_search_time

    latency_in_ms = (search_time * 1000)/queries.shape[0]

    found_distances, found_indices = cp.asnumpy(distances), cp.asnumpy(indices)
    recall = calc_recall(found_indices, gt_neighbors)

    return round(build_time_in_secs,4), round(latency_in_ms, 4), round(recall,4)

In [ ]:
ivf_pq_study = optuna.create_study(directions=['minimize', 'minimize', 'maximize'])
ivf_pq_study.optimize(multi_objective_ivf_pq, n_trials=10)

In [ ]:
print_best_trial_values(ivf_pq_study)

In [ ]:
visualization(ivf_pq_study)